# Day 65 — FastAPI for ML
### REST endpoints, Pydantic validation, async endpoints, Swagger docs, health checks

## 1. Setup

In [1]:
import sys
!{sys.executable} -m pip install fastapi uvicorn[standard] httpx scikit-learn --quiet
print("Setup complete")

Setup complete



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Why FastAPI for ML Serving?

In [2]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "framework": "Flask",
        "type": "Sync only",
        "validation": "Manual",
        "docs": "None built-in",
        "performance": "Moderate",
        "best_for": "Simple APIs, quick prototypes"
    },
    {
        "framework": "FastAPI",
        "type": "Async + Sync",
        "validation": "Automatic (Pydantic)",
        "docs": "Auto Swagger + ReDoc",
        "performance": "High (Starlette/ASGI)",
        "best_for": "Production ML APIs, async workloads"
    },
    {
        "framework": "Django REST",
        "type": "Sync only",
        "validation": "Serializers (verbose)",
        "docs": "Third-party only",
        "performance": "Moderate",
        "best_for": "Large apps with ORM, admin panel"
    },
    {
        "framework": "gRPC",
        "type": "Async",
        "validation": "Protocol Buffers",
        "docs": "Manual",
        "performance": "Very high (binary protocol)",
        "best_for": "Internal microservices, low latency"
    },
])

comparison

,framework,type,validation,docs,performance,best_for
0,Flask,Sync only,Manual,None built-in,Moderate,"Simple APIs, quick prototypes"
1,FastAPI,Async + Sync,Automatic (Pydantic),Auto Swagger + ReDoc,High (Starlette/ASGI),"Production ML APIs, async workloads"
2,Django REST,Sync only,Serializers (verbose),Third-party only,Moderate,"Large apps with ORM, admin panel"
3,gRPC,Async,Protocol Buffers,Manual,Very high (binary protocol),"Internal microservices, low latency"


## 3. Pydantic Models — Request and Response Validation

In [3]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional
from datetime import datetime
import json

# request model with validation
class PredictRequest(BaseModel):
    features: list[float] = Field(
        ...,
        min_length=1,
        max_length=100,
        description="Input features for the model"
    )
    model_version: Optional[str] = Field(
        default="latest",
        description="Model version to use"
    )

    @field_validator("features")
    @classmethod
    def features_must_be_finite(cls, v):
        import math
        for val in v:
            if not math.isfinite(val):
                raise ValueError("All features must be finite numbers (no NaN or Inf)")
        return v

# response model
class PredictResponse(BaseModel):
    prediction: float = Field(..., description="Model output value")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Confidence score (0-1)")
    model_version: str
    timestamp: datetime
    processing_time_ms: float

# error model
class ErrorResponse(BaseModel):
    error: str
    detail: str
    timestamp: datetime

# demonstrate validation
print("Valid request:")
req = PredictRequest(features=[1.0, 2.0, 3.0], model_version="v1.0")
print(f"  {req.model_dump()}")

print("\nInvalid request (empty features):")
try:
    bad_req = PredictRequest(features=[])
except Exception as e:
    print(f"  Validation error caught: {e.errors()[0]['msg']}")

print("\nInvalid request (NaN in features):")
try:
    import math
    bad_req2 = PredictRequest(features=[1.0, float('nan'), 3.0])
except Exception as e:
    print(f"  Validation error caught: {e.errors()[0]['msg']}")

Valid request:
  {'features': [1.0, 2.0, 3.0], 'model_version': 'v1.0'}

Invalid request (empty features):
  Validation error caught: List should have at least 1 item after validation, not 0

Invalid request (NaN in features):
  Validation error caught: Value error, All features must be finite numbers (no NaN or Inf)


## 4. Building the Full FastAPI ML Application

In [5]:
app_code = '''
import time
import math
from datetime import datetime
from typing import Optional
from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException, BackgroundTasks, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, field_validator
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

# Pydantic Models
class PredictRequest(BaseModel):
    features: list[float] = Field(..., min_length=1, max_length=100)
    model_version: Optional[str] = "latest"

    @field_validator("features")
    @classmethod
    def features_must_be_finite(cls, v):
        for val in v:
            if not math.isfinite(val):
                raise ValueError("All features must be finite (no NaN or Inf)")
        return v

class PredictResponse(BaseModel):
    prediction: int
    probability: float = Field(..., ge=0.0, le=1.0)
    model_version: str
    timestamp: datetime
    processing_time_ms: float

class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    model_version: str
    uptime_seconds: float
    timestamp: datetime

class BatchPredictRequest(BaseModel):
    samples: list[list[float]] = Field(..., min_length=1, max_length=100)

class BatchPredictResponse(BaseModel):
    predictions: list[int]
    probabilities: list[float]
    count: int
    processing_time_ms: float

# Global State
model = None
model_version = "1.0.0"
start_time = None
prediction_count = 0

# Lifespan (startup/shutdown)
@asynccontextmanager
async def lifespan(app: FastAPI):
    global model, start_time
    print("Loading model...")
    X, y = make_classification(n_samples=1000, n_features=4, random_state=42)
    model = LogisticRegression()
    model.fit(X, y)
    start_time = time.time()
    print("Model loaded and ready!")
    yield
    print("Shutting down...")

# App
app = FastAPI(
    title="ML Inference Service",
    description="Production-quality FastAPI ML serving API",
    version="1.0.0",
    lifespan=lifespan
)

# Middleware: request timing
@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    start = time.time()
    response = await call_next(request)
    duration = round((time.time() - start) * 1000, 2)
    response.headers["X-Process-Time-Ms"] = str(duration)
    return response

# Endpoints
@app.get("/", tags=["Root"])
def root():
    return {"message": "ML Inference Service", "docs": "/docs", "health": "/health"}

@app.get("/health", response_model=HealthResponse, tags=["Health"])
def health():
    return HealthResponse(
        status="healthy" if model is not None else "unhealthy",
        model_loaded=model is not None,
        model_version=model_version,
        uptime_seconds=round(time.time() - start_time, 1) if start_time else 0,
        timestamp=datetime.now()
    )

@app.post("/predict", response_model=PredictResponse, tags=["Inference"])
def predict(request: PredictRequest):
    global prediction_count
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")

    start = time.time()
    features = np.array(request.features).reshape(1, -1)

    if features.shape[1] != 4:
        raise HTTPException(
            status_code=422,
            detail=f"Expected 4 features, got {features.shape[1]}"
        )

    prediction = int(model.predict(features)[0])
    probability = float(model.predict_proba(features)[0][prediction])
    processing_time = round((time.time() - start) * 1000, 2)
    prediction_count += 1

    return PredictResponse(
        prediction=prediction,
        probability=probability,
        model_version=model_version,
        timestamp=datetime.now(),
        processing_time_ms=processing_time
    )

@app.post("/predict/batch", response_model=BatchPredictResponse, tags=["Inference"])
def predict_batch(request: BatchPredictRequest):
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")

    start = time.time()
    X = np.array(request.samples)
    predictions = model.predict(X).tolist()
    probabilities = [float(max(p)) for p in model.predict_proba(X)]
    processing_time = round((time.time() - start) * 1000, 2)

    return BatchPredictResponse(
        predictions=predictions,
        probabilities=probabilities,
        count=len(predictions),
        processing_time_ms=processing_time
    )

@app.get("/metrics", tags=["Monitoring"])
def metrics():
    return {
        "prediction_count": prediction_count,
        "uptime_seconds": round(time.time() - start_time, 1) if start_time else 0,
        "model_version": model_version
    }
'''

with open("ml_api.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("ml_api.py written successfully")
print(f"File size: {len(app_code)} bytes")
print("\nEndpoints defined:")
endpoints = [
    ("GET",  "/",              "Root -- service info"),
    ("GET",  "/health",        "Health check with uptime and model status"),
    ("POST", "/predict",       "Single prediction with Pydantic validation"),
    ("POST", "/predict/batch", "Batch predictions for multiple samples"),
    ("GET",  "/metrics",       "Prediction count and uptime metrics"),
]
for method, path, desc in endpoints:
    print(f"  {method:<6} {path:<22} {desc}")

ml_api.py written successfully
File size: 4669 bytes

Endpoints defined:
  GET    /                      Root -- service info
  GET    /health                Health check with uptime and model status
  POST   /predict               Single prediction with Pydantic validation
  POST   /predict/batch         Batch predictions for multiple samples
  GET    /metrics               Prediction count and uptime metrics


## 5. Start the API and Test It

In [10]:
import os
print("Current working directory:", os.getcwd())
print("ml_api.py exists:", os.path.exists("ml_api.py"))
print("Files in current dir:", [f for f in os.listdir(".") if f.endswith(".py")])

Current working directory: c:\DS-AI-75d\.vscode\week10
ml_api.py exists: True
Files in current dir: ['ml_api.py']


In [11]:
import httpx
import json

BASE = "http://127.0.0.1:8002"

print("--- GET / ---")
r = httpx.get(f"{BASE}/")
print(json.dumps(r.json(), indent=2))

print("\n--- GET /health ---")
r = httpx.get(f"{BASE}/health")
print(json.dumps(r.json(), indent=2))

print("\n--- POST /predict (valid) ---")
r = httpx.post(f"{BASE}/predict", json={"features": [0.5, -1.2, 0.8, 2.1]})
print(json.dumps(r.json(), indent=2))

print("\n--- POST /predict (wrong feature count) ---")
r = httpx.post(f"{BASE}/predict", json={"features": [1.0, 2.0]})
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

print("\n--- POST /predict/batch ---")
r = httpx.post(f"{BASE}/predict/batch", json={
    "samples": [[0.5, -1.2, 0.8, 2.1], [-0.3, 0.7, -1.5, 0.2], [1.1, 0.4, -0.8, 1.3]]
})
print(json.dumps(r.json(), indent=2))

print("\n--- GET /metrics ---")
r = httpx.get(f"{BASE}/metrics")
print(json.dumps(r.json(), indent=2))

--- GET / ---
{
  "message": "ML Inference Service",
  "docs": "/docs",
  "health": "/health"
}

--- GET /health ---
{
  "status": "healthy",
  "model_loaded": true,
  "model_version": "1.0.0",
  "uptime_seconds": 171.1,
  "timestamp": "2026-07-12T19:29:05.992437"
}

--- POST /predict (valid) ---
{
  "prediction": 1,
  "probability": 0.5467403377385356,
  "model_version": "1.0.0",
  "timestamp": "2026-07-12T19:29:06.733274",
  "processing_time_ms": 0.72
}

--- POST /predict (wrong feature count) ---
Status: 422
{
  "detail": "Expected 4 features, got 2"
}

--- POST /predict/batch ---
{
  "predictions": [
    1,
    0,
    0
  ],
  "probabilities": [
    0.5467403377385356,
    0.9046770074925559,
    0.9307935113163622
  ],
  "count": 3,
  "processing_time_ms": 0.62
}

--- GET /metrics ---
{
  "prediction_count": 1,
  "uptime_seconds": 174.0,
  "model_version": "1.0.0"
}


## 6. Summary — What We Built Today

| Feature | How FastAPI does it | Why it matters |
|---------|-------------------|----------------|
| Request validation | Pydantic BaseModel with Field constraints | Bad inputs rejected before reaching the model |
| Custom validators | @field_validator decorator | Catch NaN/Inf before numpy crashes |
| Response models | response_model= parameter | Auto-serialisation + schema in Swagger docs |
| HTTP error handling | raise HTTPException(status_code=...) | Correct HTTP status codes (422, 503) |
| Startup/shutdown | @asynccontextmanager lifespan | Model loaded once at startup, not per request |
| Middleware | @app.middleware("http") | Adds X-Process-Time-Ms header to every response |
| Auto documentation | Built-in Swagger at /docs, ReDoc at /redoc | Zero extra code -- comes from type hints |
| Batch endpoint | Separate /predict/batch endpoint | Single network call for multiple predictions |
| Health check | /health with model status + uptime | Required for Docker HEALTHCHECK and k8s probes |
| Metrics | /metrics with prediction_count | Basic observability without Prometheus |

**Key FastAPI advantages over Flask for ML:**
- Pydantic validation catches bad inputs automatically
- Swagger docs generated from type hints -- no manual documentation
- Async support handles concurrent requests efficiently
- Lifespan events load models once at startup
- 3x faster than Flask on equivalent workloads (Starlette/ASGI)